## Fake News Detector 
**Members:** Sreeram Tirumala 

**Course:** Applied Data Science

**Dataset:** ISOT Fake News Dataset (True.csv + Fake.csv)


In [5]:
!pip install seaborn matplotlib

!python3 scripts/prepare_isot.py --data_dir data/ISOT


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

===== EDA: Missing Values =====
title    0
text     0
label    0
dtype: int64
[INFO] Dropped 0 rows with missing text/label.
[INFO] EDA plots saved under 'artifacts/'.
[DONE] Wrote data/train.csv with shape (44898, 4)


In [6]:
!python3 scripts/train_sklearn.py



===== Unsupervised: TF-IDF + PCA + KMeans =====
[INFO] Saved PCA & KMeans artifacts: pca_2d.npy, pca_labels.npy, kmeans_labels.npy

===== Model 1: Logistic Regression (TF-IDF) =====
              precision    recall  f1-score   support

        fake       0.99      0.99      0.99      4696
        real       0.99      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

AUC: 0.9994
[INFO] Saved Logistic Regression model -> artifacts/sklearn_logreg_1762913119.joblib

===== Model 2: Linear SVM (TF-IDF) =====
              precision    recall  f1-score   support

        fake       1.00      0.99      1.00      4696
        real       0.99      1.00      1.00      4284

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980


===== Model 3: Random F

In [7]:
import pandas as pd
results = pd.read_csv("artifacts/model_comparison_part2.csv")
results


,model,accuracy,precision,recall,f1,roc_auc
0,LogReg TF-IDF,0.991759,0.989535,0.993231,0.991379,0.999406
1,Linear SVM TF-IDF,0.995880,0.994182,0.997199,0.995688,NaN
2,Random Forest TF-IDF,0.995880,0.994412,0.996965,0.995687,NaN


### Discussion of Results
- **Logistic Regression (TF-IDF)** achieved 99.16% accuracy with ROC-AUC 0.9994, providing an interpretable baseline.
- **Linear SVM** and **Random Forest** reached ~99.6% accuracy with slightly higher F1, showing robust generalization.
- All three methods perform excellently; however, Logistic Regression remains ideal for deployment (lightweight and explainable).

**Evaluation Metrics:**  
All metrics (accuracy, precision, recall, F1) were computed using `sklearn.metrics`.  
The ROC-AUC curve for Logistic Regression reached near perfection, confirming model reliability.


Writing & Personal Contribution — Sreeram Tirumala

I independently implemented and executed the entire Fake News Detector project using Python in VS Code. This included preparing and cleaning the ISOT dataset, performing exploratory data analysis (missing data checks, outlier detection, and correlation visualization), and generating EDA plots using Seaborn and Matplotlib.

I also integrated PCA + K-Means clustering for unsupervised analysis and implemented three supervised models — Logistic Regression, Linear SVM, and Random Forest — using scikit-learn pipelines. I compared model performances based on Accuracy, Precision, Recall, F1, and ROC-AUC metrics, and documented all results clearly.

All scripts (prepare_isot.py, train_sklearn.py) were customized, debugged, and executed solely by me. I ensured the outputs were reproducible, interpretable, and aligned with the project requirements.

This submission represents my own independent work — no external notebooks, GitHub templates, or Kaggle codebases were reused.

## Supporiting files

1. prepare_isot.py
2. train_sklearn.py

In [ ]:
import argparse, os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data_dir', required=True, help='Folder with ISOT True.csv and Fake.csv')
    ap.add_argument('--out_csv', default='data/train.csv', help='Output merged CSV')
    args = ap.parse_args()

    true_path = os.path.join(args.data_dir, 'True.csv')
    fake_path = os.path.join(args.data_dir, 'Fake.csv')

    df_true = pd.read_csv(true_path)
    df_fake = pd.read_csv(fake_path)

    # --- Assign labels and unify text/title columns ---
    for df, lbl in [(df_true, 'real'), (df_fake, 'fake')]:
        if 'text' not in df.columns:
            if 'content' in df.columns:
                df['text'] = df['content']
            else:
                raise ValueError('Could not find text/content column')
        if 'title' not in df.columns:
            df['title'] = ''
        df['label'] = lbl

    # --- Merge and shuffle ---
    df = pd.concat(
        [df_true[['title', 'text', 'label']], df_fake[['title', 'text', 'label']]],
        ignore_index=True
    )
    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

    os.makedirs(os.path.dirname(args.out_csv), exist_ok=True)
    os.makedirs("artifacts", exist_ok=True)

    plt.rcParams['figure.figsize'] = (8, 5)
    sns.set(style="whitegrid")

    print("\n===== EDA: Missing Values =====")
    print(df.isnull().sum())

    # Missing-values heatmap
    sns.heatmap(df.isnull(), cbar=False)
    plt.title("Missing Values Heatmap")
    plt.tight_layout()
    plt.savefig("artifacts/missing_heatmap.png")
    plt.close()

    # Drop rows with missing text/label
    before = len(df)
    df = df.dropna(subset=['text', 'label'])
    after = len(df)
    print(f"[INFO] Dropped {before - after} rows with missing text/label.")

    # Label distribution
    sns.countplot(x='label', data=df)
    plt.title("Label Distribution")
    plt.tight_layout()
    plt.savefig("artifacts/label_distribution.png")
    plt.close()

    # Text-length analysis
    df['text_len'] = df['text'].astype(str).str.len()
    sns.boxplot(x=df['text_len'])
    plt.title("Text Length Distribution")
    plt.tight_layout()
    plt.savefig("artifacts/text_length_box.png")
    plt.close()

    sns.histplot(df['text_len'], bins=50)
    plt.title("Text Length Histogram")
    plt.xlabel("Characters per article")
    plt.tight_layout()
    plt.savefig("artifacts/text_length_hist.png")
    plt.close()

    print("[INFO] EDA plots saved under 'artifacts/'.")

    df.to_csv(args.out_csv, index=False)
    print(f"[DONE] Wrote {args.out_csv} with shape {df.shape}")

if __name__ == '__main__':
    main()


In [ ]:
import argparse, os, yaml, time, json
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, roc_curve, precision_recall_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score
)

from utils.io import save_artifact


def load_config(path: str):
    with open(path, "r") as f:
        return yaml.safe_load(f)


def build_tfidf(cfg):
    return TfidfVectorizer(
        max_features=cfg["sklearn"]["max_features"],
        ngram_range=tuple(cfg["sklearn"]["ngram_range"]),
        stop_words="english"
    )


def evaluate_row(name, y_true, y_pred, y_proba=None):
    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": None,
    }
    if y_proba is not None:
        try:
            row["roc_auc"] = roc_auc_score(y_true, y_proba)
        except ValueError:
            row["roc_auc"] = None
    return row


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="config.yaml")
    ap.add_argument("--out_dir", default="artifacts")
    args = ap.parse_args()

    cfg = load_config(args.config)
    os.makedirs(args.out_dir, exist_ok=True)

    # ========================= Load data =========================
    data = pd.read_csv(cfg["dataset"]["train_csv"])

    text_col = cfg["dataset"]["text_col"]
    title_col = cfg["dataset"].get("title_col") or ""
    label_col = cfg["dataset"]["label_col"]
    pos_label = cfg["dataset"]["positive_label"]

    if title_col and title_col in data.columns:
        X_full = (data[title_col].fillna("") + " " + data[text_col].fillna("")).astype(str)
    else:
        X_full = data[text_col].astype(str)

    y_full = (data[label_col] == pos_label).astype(int)

    # ==================== Unsupervised: PCA + KMeans ====================
    print("\n===== Unsupervised: TF-IDF + PCA + KMeans =====")

    tfidf_unsup = build_tfidf(cfg)
    X_tfidf_full = tfidf_unsup.fit_transform(X_full)

    svd = TruncatedSVD(n_components=2, random_state=42)
    X_2d = svd.fit_transform(X_tfidf_full)

    kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
    cluster_labels = kmeans.fit_predict(X_tfidf_full)

    np.save(os.path.join(args.out_dir, "pca_2d.npy"), X_2d)
    np.save(os.path.join(args.out_dir, "pca_labels.npy"), data[label_col].values)
    np.save(os.path.join(args.out_dir, "kmeans_labels.npy"), cluster_labels)

    print("[INFO] Saved PCA & KMeans artifacts: pca_2d.npy, pca_labels.npy, kmeans_labels.npy")

    # ==================== Train/Test Split ====================
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_full, y_full,
        test_size=cfg["sklearn"]["test_size"],
        random_state=cfg["sklearn"]["random_state"],
        stratify=y_full
    )

    results = []

    # ==================== Model 1: Logistic Regression ====================
    print("\n===== Model 1: Logistic Regression (TF-IDF) =====")

    logreg_pipe = Pipeline([
        ("tfidf", build_tfidf(cfg)),
        ("clf", LogisticRegression(
            C=cfg["sklearn"]["C"],
            penalty=cfg["sklearn"]["penalty"],
            solver=cfg["sklearn"]["solver"],
            max_iter=1000
        ))
    ])

    logreg_pipe.fit(X_tr, y_tr)
    y_pred_lr = logreg_pipe.predict(X_te)
    y_proba_lr = logreg_pipe.predict_proba(X_te)[:, 1]

    print(classification_report(y_te, y_pred_lr, target_names=["fake", "real"]))
    try:
        print(f"AUC: {roc_auc_score(y_te, y_proba_lr):.4f}")
    except Exception:
        pass

    cm = confusion_matrix(y_te, y_pred_lr).tolist()
    fpr, tpr, _ = roc_curve(y_te, y_proba_lr)
    prec, rec, _ = precision_recall_curve(y_te, y_proba_lr)
    curves = {
        "roc": {"fpr": fpr.tolist(), "tpr": tpr.tolist(), "auc": float(auc(fpr, tpr))},
        "pr": {"precision": prec.tolist(), "recall": rec.tolist()},
    }

    ts = int(time.time())
    lr_model_path = os.path.join(args.out_dir, f"sklearn_logreg_{ts}.joblib")
    save_artifact(logreg_pipe, lr_model_path)

    with open(os.path.join(args.out_dir, f"metrics_{ts}.json"), "w") as fh:
        json.dump({
            "report": classification_report(y_te, y_pred_lr, output_dict=True),
            "confusion_matrix": cm,
            "curves": curves,
        }, fh, indent=2)

    print(f"[INFO] Saved Logistic Regression model -> {lr_model_path}")
    results.append(evaluate_row("LogReg TF-IDF", y_te, y_pred_lr, y_proba_lr))

    # ==================== Model 2: Linear SVM ====================
    print("\n===== Model 2: Linear SVM (TF-IDF) =====")

    svm_pipe = Pipeline([
        ("tfidf", build_tfidf(cfg)),
        ("clf", LinearSVC())
    ])

    svm_pipe.fit(X_tr, y_tr)
    y_pred_svm = svm_pipe.predict(X_te)

    print(classification_report(y_te, y_pred_svm, target_names=["fake", "real"]))
    results.append(evaluate_row("Linear SVM TF-IDF", y_te, y_pred_svm))

    # ==================== Model 3: Random Forest ====================
    print("\n===== Model 3: Random Forest (TF-IDF) =====")

    rf_pipe = Pipeline([
        ("tfidf", build_tfidf(cfg)),
        ("clf", RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            n_jobs=-1,
            random_state=42
        ))
    ])

    rf_pipe.fit(X_tr, y_tr)
    y_pred_rf = rf_pipe.predict(X_te)

    print(classification_report(y_te, y_pred_rf, target_names=["fake", "real"]))
    results.append(evaluate_row("Random Forest TF-IDF", y_te, y_pred_rf))

    # ==================== Save comparison ====================
    results_df = pd.DataFrame(results)
    comp_csv = os.path.join(args.out_dir, "model_comparison_part2.csv")
    comp_json = os.path.join(args.out_dir, "model_comparison_part2.json")

    results_df.to_csv(comp_csv, index=False)
    with open(comp_json, "w") as f:
        json.dump(results, f, indent=2)

    print("\n===== Model Comparison (Part 2) =====")
    print(results_df)
    print(f"[INFO] Saved comparison to {comp_csv}")
    print("[DONE] Training + evaluation complete.")


if __name__ == "__main__":
    main()


![alt text](image.png)